# Magic Pen (Air Gesture Drawing) - Gesture Data Collection

In this project, we are creating an **Air Gesture Drawing** system, where the user can draw in the air using hand gestures.  
Before we start collecting data for training, we need to **verify that each hand gesture can be detected by MediaPipe**.  

This step ensures that every gesture we record has a **hand landmark** available.  
If MediaPipe can detect the hand and its landmarks correctly, we can safely save the gesture data for later use in training our model.  

### What we will do:
1. Capture hand gestures using a webcam.
2. Use MediaPipe to detect hand landmarks.
3. Display the landmarks on the screen to verify detection.
4. Only collect gesture data if landmarks are properly detected.

> This preliminary check helps ensure the quality and reliability of the dataset for our Air Gesture Drawing project.

In [1]:
import cv2
import mediapipe as mp

# Initialize MediaPipe Hands
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(
    static_image_mode=False,   # Use live feed
    max_num_hands=2,           # Detect up to 2 hands
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

# Open webcam
cap = cv2.VideoCapture(0)

print("Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Flip the image horizontally for natural interaction
    frame = cv2.flip(frame, 1)

    # Convert BGR to RGB
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Process the frame to detect hands
    result = hands.process(rgb_frame)

    # Draw hand landmarks if detected
    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
        cv2.putText(frame, "Hand Detected!", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    else:
        cv2.putText(frame, "No Hand Detected", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    # Show the frame
    cv2.imshow("Hand Landmark Verification", frame)

    # Quit when 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
cv2.destroyAllWindows()
hands.close()

Press 'q' to quit.


<div style="text-align: center;">
  <video width="1000" height="500" controls>
    <source src="videos/demo_hand_Detected.mp4" type="video/mp4">
  </video>
</div>


Since each gesture is successfully detected by MediaPipe and provides 21 hand landmarks, we can now proceed to **collect data for each gesture**.  
This data will be used as **training data** for our Air Gesture Drawing model.

In [2]:
import cv2
import mediapipe as mp
import csv
import os
import time

gestures = ["Draw", "Erase", "Scroll", "Cursor","Color","Screenshot","Clear"]
samples_per_gesture = 600
output_csv = "gesture_data2.csv"

# Create CSV file
if not os.path.exists(output_csv):

    header = ["label"]

    # Left hand
    for i in range(21):
        header += [f"Lx{i}", f"Ly{i}", f"Lz{i}"]

    # Right hand
    for i in range(21):
        header += [f"Rx{i}", f"Ry{i}", f"Rz{i}"]

    with open(output_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(header)

# MediaPipe setup
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

cap = cv2.VideoCapture(0)
time.sleep(2)

print("Press 'q' anytime to quit.\n")

# Start gesture collection
for label, gesture_name in enumerate(gestures):

    print(f"Prepare to perform gesture: {gesture_name}")

    for wait_sec in range(5,0,-1):
        print(f"Starting in {wait_sec} seconds...")
        time.sleep(1)

    collected_frames = 0

    while collected_frames < samples_per_gesture:

        ret, frame = cap.read()
        if not ret:
            continue

        frame = cv2.flip(frame,1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)

        # default values if hands not detected
        left_hand = [0]*63
        right_hand = [0]*63

        if result.multi_hand_landmarks and result.multi_handedness:

            for idx, hand_landmarks in enumerate(result.multi_hand_landmarks):

                hand_label = result.multi_handedness[idx].classification[0].label

                mp_drawing.draw_landmarks(
                    frame,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS
                )

                landmarks = []
                for lm in hand_landmarks.landmark:
                    landmarks += [lm.x, lm.y, lm.z]

                if hand_label == "Left":
                    left_hand = landmarks
                else:
                    right_hand = landmarks

            # save row
            row = [label] + left_hand + right_hand

            with open(output_csv, "a", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(row)

            collected_frames += 1

        cv2.putText(
            frame,
            f"{gesture_name} ({collected_frames}/{samples_per_gesture})",
            (10,30),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0,255,0),
            2
        )

        cv2.imshow("Collecting Gesture Data", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    print(f"\nFinished collecting data for gesture: {gesture_name}")

    if label < len(gestures)-1:
        print(f"Next gesture: {gestures[label+1]}")
        print("Prepare for next gesture...")
        time.sleep(5)

print("\nData collection complete!")

cap.release()
cv2.destroyAllWindows()
hands.close()

Press 'q' anytime to quit.

Prepare to perform gesture: Draw
Starting in 5 seconds...
Starting in 4 seconds...
Starting in 3 seconds...
Starting in 2 seconds...
Starting in 1 seconds...

Finished collecting data for gesture: Draw
Next gesture: Erase
Prepare for next gesture...
Prepare to perform gesture: Erase
Starting in 5 seconds...
Starting in 4 seconds...
Starting in 3 seconds...
Starting in 2 seconds...
Starting in 1 seconds...

Finished collecting data for gesture: Erase
Next gesture: Scroll
Prepare for next gesture...
Prepare to perform gesture: Scroll
Starting in 5 seconds...
Starting in 4 seconds...
Starting in 3 seconds...
Starting in 2 seconds...
Starting in 1 seconds...

Finished collecting data for gesture: Scroll
Next gesture: Cursor
Prepare for next gesture...
Prepare to perform gesture: Cursor
Starting in 5 seconds...
Starting in 4 seconds...
Starting in 3 seconds...
Starting in 2 seconds...
Starting in 1 seconds...

Finished collecting data for gesture: Cursor
Next ges

<div style="text-align: center;">
  <video width="1000" height="500" controls>
    <source src="videos/data_collection.mp4" type="video/mp4">
  </video>
</div>


The collected dataset contains hand gesture landmark coordinates obtained using MediaPipe. Each gesture sample includes the x, y, and z coordinates of 21 hand landmarks detected from the webcam. These coordinates are stored in a CSV file along with a label representing the gesture type, which will be used as training data for the gesture recognition model.
### Gesture Labels

| Label | Gesture Name |
|------|-------------|
| 0 | Draw |
| 1 | Erase |
| 2 | Scroll |
| 3 | Cursor |
| 4 | Color |
| 5 | Screenshot |
| 6 | Clear |

### Data Description

In [1]:
import pandas as pd
df=pd.read_csv("gesture_data.csv")
df.head()

,label,Lx0,Ly0,Lz0,Lx1,Ly1,Lz1,Lx2,Ly2,Lz2,...,Rz17,Rx18,Ry18,Rz18,Rx19,Ry19,Rz19,Rx20,Ry20,Rz20
0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.024006,0.640282,0.638152,-0.043547,0.636939,0.677689,-0.036370,0.644005,0.714818,-0.025161
1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.017363,0.644825,0.643655,-0.034172,0.642093,0.680825,-0.028700,0.646941,0.717298,-0.019838
2,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.002029,0.611983,0.575341,-0.008682,0.607034,0.606629,-0.000598,0.612013,0.638193,0.008795
3,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.007133,0.551270,0.492378,0.004686,0.545436,0.519785,0.013260,0.552976,0.545225,0.022690
4,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.007748,0.548172,0.460406,-0.020481,0.540783,0.504239,-0.011554,0.552928,0.537528,0.000851


In [2]:
df.tail()

,label,Lx0,Ly0,Lz0,Lx1,Ly1,Lz1,Lx2,Ly2,Lz2,...,Rz17,Rx18,Ry18,Rz18,Rx19,Ry19,Rz19,Rx20,Ry20,Rz20
1395,6,0.207213,0.878242,5.308571e-07,0.266424,0.832545,-0.023577,0.307283,0.763719,-0.035858,...,-0.040315,0.799556,0.544404,-0.054920,0.805850,0.501157,-0.060824,0.810396,0.460338,-0.065245
1396,6,0.204545,0.878549,5.174877e-07,0.264881,0.833900,-0.024081,0.306507,0.763990,-0.035903,...,-0.040893,0.799200,0.544858,-0.055506,0.805443,0.501258,-0.061380,0.809840,0.459901,-0.065803
1397,6,0.203516,0.877610,5.299175e-07,0.264300,0.832461,-0.023838,0.306240,0.764741,-0.035308,...,-0.040864,0.798230,0.544708,-0.055356,0.804499,0.501161,-0.060780,0.808841,0.460741,-0.064790
1398,6,0.203408,0.878715,5.312584e-07,0.263341,0.834594,-0.024173,0.304502,0.765993,-0.035561,...,-0.042254,0.796497,0.547194,-0.057179,0.802521,0.502834,-0.063247,0.806606,0.460825,-0.067797
1399,6,0.201601,0.878884,5.076827e-07,0.262130,0.832564,-0.022510,0.303807,0.763445,-0.033324,...,-0.044896,0.795085,0.547266,-0.059963,0.801080,0.502549,-0.065945,0.805102,0.460673,-0.070332


In [3]:
df.shape

(1400, 127)

In [4]:
df.duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
1395    False
1396    False
1397    False
1398    False
1399    False
Length: 1400, dtype: bool

In [7]:
label_counts = df["label"].value_counts()

print(label_counts)

label
0    200
1    200
2    200
3    200
4    200
5    200
6    200
Name: count, dtype: int64
